# Agent Middleware Types

Core types, state schemas, base classes, and decorators used to build LangChain agent middleware.

This module provides:

* Request and response containers for model calls.
* Standard agent input, output, and internal state schemas.
* Schema-omission annotations for middleware state fields.
* The `AgentMiddleware` base class and lifecycle hooks.
* Decorators that convert standalone functions into middleware instances.
* Re-exported tool-call request and wrapper types from LangGraph.

# `JumpTo`

Type alias describing valid destinations for middleware-controlled graph jumps.

```python
JumpTo = Literal[
    "tools",
    "model",
    "end"
]
```

## Values

* `"tools"` — Jump to the tool-execution node.
* `"model"` — Jump back to the model node.
* `"end"` — End the agent graph.

A middleware hook that may return one of these destinations should declare it using `hook_config` or the `can_jump_to` parameter of a lifecycle decorator.

# `ResponseT`

Generic type variable representing structured model output.

```python
ResponseT = TypeVar(
    "ResponseT",
    default=Any
)
```

When no structured-output type is specified, it defaults to `Any`.

# `ContextT`

Runtime-context type variable imported from LangGraph.

It is used throughout middleware types such as:

```python
Runtime[ContextT]
ModelRequest[ContextT]
AgentMiddleware[StateT, ContextT, ResponseT]
```

# `ModelRequest`

Dataclass containing all information needed for one model call.

- Bases: `Generic[ContextT]`
- Decorator: `@dataclass(init=False)`

The custom constructor preserves backward compatibility with the deprecated `system_prompt` argument.

## Constructor

```python
ModelRequest(
    *,
    model: BaseChatModel,
    messages: list[AnyMessage],
    system_message: SystemMessage | None = None,
    system_prompt: str | None = None,
    tool_choice: Any | None = None,
    tools: list[
        BaseTool | dict[str, Any]
    ] | None = None,
    response_format: ResponseFormat[Any] | None = None,
    state: AgentState[Any] | None = None,
    runtime: Runtime[ContextT] | None = None,
    model_settings: dict[str, Any] | None = None
)
```

## Parameters

* `model` — Chat model that should process the request.
* `messages` — Conversation messages excluding the system message.
* `system_message` — Preferred complete system-message object.
* `system_prompt` — Deprecated system-prompt string.
  * Converted internally to `SystemMessage(content=system_prompt)`.
  * Cannot be provided together with `system_message`.
* `tool_choice` — Provider or model-specific tool-choice configuration.
* `tools` — Tools available during the model call.
  * Defaults to an empty list.
  * May contain `BaseTool` instances or provider-native tool dictionaries.
* `response_format` — Optional structured-output specification.
* `state` — Current agent state.
  * Defaults to:
    ```python
    {"messages": []}
    ```
* `runtime` — LangGraph runtime carrying context, configuration, and streaming facilities.
* `model_settings` — Additional model-call settings.
  * Defaults to an empty dictionary.

## Attributes

```python
model: BaseChatModel
messages: list[AnyMessage]
system_message: SystemMessage | None
tool_choice: Any | None
tools: list[BaseTool | dict[str, Any]]
response_format: ResponseFormat[Any] | None
state: AgentState[Any]
runtime: Runtime[ContextT]
model_settings: dict[str, Any]
```

## Property: `system_prompt`

Returns the text extracted from `system_message`.

```python
@property
def system_prompt(
    self
) -> str | None
```

Behaviour:

```text
system_message is None -> None
system_message exists  -> system_message.text
```

The property exists for backward compatibility.

## Direct Attribute Assignment

Direct assignment is deprecated:

```python
request.model = another_model
request.tools = new_tools
request.system_prompt = "New prompt"
```

Every direct assignment emits `DeprecationWarning`.

Use:

```python
new_request = request.override(
    model=another_model
)
```

instead.

Assigning `system_prompt` receives special handling:

```python
request.system_prompt = None
```

sets:

```python
request.system_message = None
```

and assigning a string creates a new `SystemMessage`.

The constructor suppresses these warnings while initializing the dataclass fields internally.

## Method: `override`

Creates a new request containing selected replacements while leaving the original request unchanged.

```python
override(
    self,
    **overrides: Unpack[
        _ModelRequestOverrides
    ]
) -> ModelRequest[ContextT]
```

## Supported Overrides

```python
model: BaseChatModel
system_message: SystemMessage | None
messages: list[AnyMessage]
tool_choice: Any | None
tools: list[BaseTool | dict[str, Any]]
response_format: ResponseFormat[Any] | None
model_settings: dict[str, Any]
state: AgentState[Any]
```

The deprecated `system_prompt` key is also accepted at runtime.

## Examples

```python
new_request = request.override(
    model=different_model
)
```

```python
from langchain_core.messages import SystemMessage

new_request = request.override(
    system_message=SystemMessage(
        content="Answer concisely."
    )
)
```

```python
new_request = request.override(
    tools=[search_tool],
    model_settings={
        "temperature": 0
    }
)
```

## Validation

Both the constructor and `override` raise:

```python
ValueError(
    "Cannot specify both system_prompt and system_message"
)
```

when both forms are provided.

# `_ModelRequestOverrides`

Internal `TypedDict` describing supported `ModelRequest.override` keys.

```python
class _ModelRequestOverrides(
    TypedDict,
    total=False
):
    model: BaseChatModel
    system_message: SystemMessage | None
    messages: list[AnyMessage]
    tool_choice: Any | None
    tools: list[
        BaseTool | dict[str, Any]
    ]
    response_format: ResponseFormat[Any] | None
    model_settings: dict[str, Any]
    state: AgentState[Any]
```

It is internal and is not exported through `__all__`.

# `ModelResponse`

Dataclass representing the result of model execution.

- Bases: `Generic[ResponseT]`

```python
@dataclass
class ModelResponse(
    Generic[ResponseT]
):
    result: list[BaseMessage]
    structured_response: ResponseT | None = None
```

## Attributes

* `result` — Messages produced during model execution.
  * Usually contains one `AIMessage`.
  * May also contain a `ToolMessage` when a tool is used for structured output.
* `structured_response` — Parsed structured output.
  * `None` when no response format was configured.

# `ExtendedModelResponse`

Dataclass that combines a model response with an optional LangGraph `Command`.

- Bases: `Generic[ResponseT]`

```python
@dataclass
class ExtendedModelResponse(
    Generic[ResponseT]
):
    model_response: ModelResponse[ResponseT]
    command: Command[Any] | None = None
```

## Attributes

* `model_response` — Underlying model result.
* `command` — Additional state update to apply after the model node finishes.

## Reducer Behaviour

The command is applied through the graph's reducers.

For fields using reducers, such as:

```python
messages: Annotated[
    list[AnyMessage],
    add_messages
]
```

command messages are added alongside model-response messages rather than replacing them.

For state fields without reducers, later commands overwrite earlier values. With nested model-call middleware, the outermost middleware's later update wins.

## Unsupported Command Features

At this source revision, commands returned through `ExtendedModelResponse` do not support:

```text
goto
resume
graph
```

Using those command features raises `NotImplementedError`.

# `ModelCallResult`

Union type accepted as a middleware model-call result.

```python
ModelCallResult = (
    ModelResponse[ResponseT]
    | AIMessage
    | ExtendedModelResponse[ResponseT]
)
```

## Variants

* `ModelResponse` — Complete result with optional structured output.
* `AIMessage` — Convenience result for simple middleware.
* `ExtendedModelResponse` — Model result plus an additional state-update command.

# `OmitFromSchema`

Annotation used to omit middleware state fields from generated input or output schemas.

```python
@dataclass
class OmitFromSchema:
    input: bool = True
    output: bool = True
```

## Attributes

* `input` — Whether to omit the field from the input schema.
* `output` — Whether to omit the field from the output schema.

## Predefined Annotations

### `OmitFromInput`

```python
OmitFromInput = OmitFromSchema(
    input=True,
    output=False
)
```

The field is hidden from the agent's input schema but may appear in output.

### `OmitFromOutput`

```python
OmitFromOutput = OmitFromSchema(
    input=False,
    output=True
)
```

The field may be accepted as input but is hidden from output.

### `PrivateStateAttr`

```python
PrivateStateAttr = OmitFromSchema(
    input=True,
    output=True
)
```

The field is internal to middleware and omitted from both external schemas.

## Example

```python
class CustomState(
    AgentState[Any]
):
    internal_counter: NotRequired[
        Annotated[
            int,
            PrivateStateAttr
        ]
    ]
```

# `AgentState`

Standard internal state schema for an agent.

- Bases: `TypedDict`
- Generic over: `ResponseT`

```python
class AgentState(
    TypedDict,
    Generic[ResponseT]
):
    messages: Required[
        Annotated[
            list[AnyMessage],
            add_messages
        ]
    ]

    jump_to: NotRequired[
        Annotated[
            JumpTo | None,
            EphemeralValue,
            PrivateStateAttr
        ]
    ]

    structured_response: NotRequired[
        Annotated[
            ResponseT,
            OmitFromInput
        ]
    ]
```

## Fields

* `messages` — Required message history.
  * Uses the LangGraph `add_messages` reducer.
* `jump_to` — Optional temporary graph destination.
  * Uses `EphemeralValue`.
  * Omitted from both input and output schemas.
* `structured_response` — Optional parsed structured output.
  * Omitted from the input schema.

# `InputAgentState`

External input-state schema.

```python
class InputAgentState(
    TypedDict
):
    messages: Required[
        Annotated[
            list[
                AnyMessage
                | dict[str, Any]
            ],
            add_messages
        ]
    ]
```

Unlike internal `AgentState`, input messages may be supplied as message objects or message dictionaries.

# `OutputAgentState`

External output-state schema.

- Generic over: `ResponseT`

```python
class OutputAgentState(
    TypedDict,
    Generic[ResponseT]
):
    messages: Required[
        Annotated[
            list[AnyMessage],
            add_messages
        ]
    ]

    structured_response: NotRequired[
        ResponseT
    ]
```

# Deprecated State Aliases

The module preserves two private names for backward compatibility:

```python
_InputAgentState = InputAgentState
_OutputAgentState = OutputAgentState
```

They are intended for removal in a future release.

# State Type Variables

```python
StateT = TypeVar(
    "StateT",
    bound=AgentState[Any],
    default=AgentState[Any]
)
```

```python
StateT_co = TypeVar(
    "StateT_co",
    bound=AgentState[Any],
    default=AgentState[Any],
    covariant=True
)
```

```python
StateT_contra = TypeVar(
    "StateT_contra",
    bound=AgentState[Any],
    contravariant=True
)
```

`StateT_co` is exported. `StateT` and `StateT_contra` are internal typing helpers at this revision.

# `AgentMiddleware`

Base class for creating agent middleware.

```python
class AgentMiddleware(
    Generic[
        StateT,
        ContextT,
        ResponseT
    ]
)
```

## Type Parameters

* `StateT` — Agent-state schema.
  * Defaults to `AgentState[Any]`.
* `ContextT` — Runtime-context type.
  * Defaults through LangGraph's context type.
* `ResponseT` — Structured-response type.
  * Defaults to `Any`.

## Class Attributes

### `state_schema`

```python
state_schema: type[StateT]
```

Schema passed to middleware nodes.

The default is the internal `_DefaultAgentState`, which extends `AgentState[Any]`.

### `tools`

```python
tools: Sequence[BaseTool]
```

Additional tools registered by the middleware.

Subclasses that add no tools generally use:

```python
tools = []
```

### `transformers`

```python
transformers: Sequence[
    TransformerFactory
] = ()
```

Stream-transformer factories registered by the middleware.

Each factory is invoked with a stream scope so every invocation receives a fresh transformer instance.

Factories are merged at graph-compilation time:

```text
ToolCallTransformer
        |
middleware transformers
        |
user-supplied transformers
```

## Property: `name`

Returns the middleware instance's class name.

```python
@property
def name(
    self
) -> str
```

A subclass may override this property to provide instance-specific names.

# Lifecycle Hooks

## `before_agent`

Runs once before agent execution begins.

```python
before_agent(
    self,
    state: StateT,
    runtime: Runtime[ContextT]
) -> dict[str, Any] | None
```

The base implementation performs no work and implicitly returns `None`.

## `abefore_agent`

Asynchronous pre-agent hook.

```python
async def abefore_agent(
    self,
    state: StateT,
    runtime: Runtime[ContextT]
) -> dict[str, Any] | None
```

The base implementation performs no work.

## `before_model`

Runs before each model call.

```python
before_model(
    self,
    state: StateT,
    runtime: Runtime[ContextT]
) -> dict[str, Any] | None
```

## `abefore_model`

Asynchronous pre-model hook.

```python
async def abefore_model(
    self,
    state: StateT,
    runtime: Runtime[ContextT]
) -> dict[str, Any] | None
```

## `after_model`

Runs after each model call.

```python
after_model(
    self,
    state: StateT,
    runtime: Runtime[ContextT]
) -> dict[str, Any] | None
```

## `aafter_model`

Asynchronous post-model hook.

```python
async def aafter_model(
    self,
    state: StateT,
    runtime: Runtime[ContextT]
) -> dict[str, Any] | None
```

## `after_agent`

Runs after agent execution completes.

```python
after_agent(
    self,
    state: StateT,
    runtime: Runtime[ContextT]
) -> dict[str, Any] | None
```

## `aafter_agent`

Asynchronous post-agent hook.

```python
async def aafter_agent(
    self,
    state: StateT,
    runtime: Runtime[ContextT]
) -> dict[str, Any] | None
```

# Model-Call Wrappers

## `wrap_model_call`

Intercepts synchronous model execution.

```python
wrap_model_call(
    self,
    request: ModelRequest[ContextT],
    handler: Callable[
        [ModelRequest[ContextT]],
        ModelResponse[ResponseT]
    ]
) -> (
    ModelResponse[ResponseT]
    | AIMessage
    | ExtendedModelResponse[ResponseT]
)
```

Middleware may:

* Modify the request through `request.override`.
* Call the handler once.
* Call the handler repeatedly for retry or fallback logic.
* Skip the handler to short-circuit execution.
* Rewrite the response.
* Return a plain `AIMessage`.
* Return an `ExtendedModelResponse`.

Multiple middleware compose with the first middleware in the list as the outermost wrapper.

The base implementation raises `NotImplementedError` with instructions for resolving a sync/async mismatch.

## `awrap_model_call`

Intercepts asynchronous model execution.

```python
async def awrap_model_call(
    self,
    request: ModelRequest[ContextT],
    handler: Callable[
        [ModelRequest[ContextT]],
        Awaitable[
            ModelResponse[ResponseT]
        ]
    ]
) -> (
    ModelResponse[ResponseT]
    | AIMessage
    | ExtendedModelResponse[ResponseT]
)
```

The base implementation raises `NotImplementedError` when only the synchronous wrapper was implemented but the agent is invoked asynchronously.

# Tool-Call Wrappers

## `wrap_tool_call`

Intercepts synchronous tool execution.

```python
wrap_tool_call(
    self,
    request: ToolCallRequest,
    handler: Callable[
        [ToolCallRequest],
        ToolMessage | Command[Any]
    ]
) -> ToolMessage | Command[Any]
```

Middleware may:

* Inspect the tool call, state, and runtime.
* Modify the call using `request.override`.
* Call the handler repeatedly for retries.
* Skip execution and return a cached or synthetic result.
* Rewrite the final `ToolMessage` or `Command`.

Each handler call is independent and stateless.

Exceptions propagate unless tool-error handling is configured on the `ToolNode`.

The base implementation raises `NotImplementedError` for a sync/async mismatch.

## `awrap_tool_call`

Asynchronous tool-call wrapper.

```python
async def awrap_tool_call(
    self,
    request: ToolCallRequest,
    handler: Callable[
        [ToolCallRequest],
        Awaitable[
            ToolMessage | Command[Any]
        ]
    ]
) -> ToolMessage | Command[Any]
```

The base implementation raises `NotImplementedError` when the middleware lacks an asynchronous implementation.

# `hook_config`

Decorator that adds graph-jump metadata to a middleware hook.

```python
hook_config(
    *,
    can_jump_to: list[JumpTo] | None = None
) -> Callable[
    [CallableT],
    CallableT
]
```

## Parameters

* `can_jump_to` — Destinations that the decorated hook may return.
  * `"tools"`
  * `"model"`
  * `"end"`

## Behaviour

The decorator stores the values on the function as:

```python
func.__can_jump_to__ = can_jump_to
```

The lifecycle decorators read and preserve this metadata on their generated hook methods.

## Example

```python
class StopMiddleware(
    AgentMiddleware
):
    @hook_config(
        can_jump_to=["end"]
    )
    def before_model(
        self,
        state,
        runtime
    ):
        if should_stop(state):
            return {
                "jump_to": "end"
            }

        return None
```

# Lifecycle Function Decorators

The following decorators dynamically create an `AgentMiddleware` subclass and immediately instantiate it:

```text
before_agent
before_model
after_model
after_agent
```

They support both forms:

```python
@before_model
def middleware(...):
    ...
```

and:

```python
@before_model(
    state_schema=CustomState,
    tools=[tool],
    can_jump_to=["end"],
    name="CustomMiddleware"
)
def middleware(...):
    ...
```

## Shared Signature

```python
decorator(
    func=None,
    *,
    state_schema: type[StateT] | None = None,
    tools: list[BaseTool] | None = None,
    can_jump_to: list[JumpTo] | None = None,
    name: str | None = None
)
```

## Shared Parameters

* `func` — Synchronous or asynchronous hook function.
* `state_schema` — Custom middleware-state schema.
  * Defaults to `AgentState`.
* `tools` — Additional tools registered by the generated middleware.
  * Defaults to an empty list.
* `can_jump_to` — Valid graph destinations.
* `name` — Generated middleware-class name.
  * Defaults to the decorated function's name.

## Function Signature

The decorated function must accept:

```python
(
    state: StateT,
    runtime: Runtime[ContextT]
)
```

and may return:

```python
dict[str, Any]
| Command[Any]
| None
```

## Sync/Async Generation

For a synchronous function, the generated class implements only the synchronous hook:

```text
before_agent
before_model
after_model
after_agent
```

For an asynchronous function, it implements only the matching asynchronous hook:

```text
abefore_agent
abefore_model
aafter_model
aafter_agent
```

Unlike `dynamic_prompt`, these decorators do not automatically create an async wrapper around a synchronous lifecycle function.

# `before_model`

Creates middleware whose hook runs before every model call.

```python
before_model(
    func=None,
    *,
    state_schema=None,
    tools=None,
    can_jump_to=None,
    name=None
)
```

## Example

```python
from langchain.agents.middleware import (
    AgentState,
    before_model
)
from langgraph.runtime import Runtime

@before_model
def log_before_model(
    state: AgentState,
    runtime: Runtime
) -> None:
    print(
        len(state["messages"])
    )
```

## Conditional Jump

```python
@before_model(
    can_jump_to=["end"]
)
def stop_if_needed(
    state,
    runtime
):
    if should_stop(state):
        return {
            "jump_to": "end"
        }

    return None
```

# `after_model`

Creates middleware whose hook runs after every model call.

```python
after_model(
    func=None,
    *,
    state_schema=None,
    tools=None,
    can_jump_to=None,
    name=None
)
```

## Example

```python
@after_model
def log_response(
    state,
    runtime
):
    print(
        state["messages"][-1]
    )
```

# `before_agent`

Creates middleware whose hook runs before the complete agent execution.

```python
before_agent(
    func=None,
    *,
    state_schema=None,
    tools=None,
    can_jump_to=None,
    name=None
)
```

## Example

```python
@before_agent
def initialize_state(
    state,
    runtime
):
    return {
        "request_count": 0
    }
```

# `after_agent`

Creates middleware whose hook runs after agent execution completes.

```python
after_agent(
    func=None,
    *,
    state_schema=None,
    tools=None,
    can_jump_to=None,
    name=None
)
```

## Example

```python
@after_agent
def record_completion(
    state,
    runtime
):
    save_result(state)
```

# Streaming Custom Events

Lifecycle decorators receive the LangGraph runtime and can emit custom events through:

```python
runtime.stream_writer(
    {
        "type": "status",
        "message": "Thinking..."
    }
)
```

These events can be consumed when using:

```python
stream_mode="custom"
```

or a list containing `"custom"`.

# `dynamic_prompt`

Decorator that creates middleware for generating the model system message dynamically.

```python
dynamic_prompt(
    func=None
)
```

## Decorated Function Signature

```python
def prompt_function(
    request: ModelRequest[ContextT]
) -> str | SystemMessage:
    ...
```

The function may also be asynchronous.

## Behaviour

1. Calls the decorated function with the current `ModelRequest`.
2. Accepts either:
   * A prompt string.
   * A complete `SystemMessage`.
3. Converts a string into:
   ```python
   SystemMessage(content=prompt)
   ```
4. Creates a new request using:
   ```python
   request.override(
       system_message=...
   )
   ```
5. Calls the next model handler.

## Sync/Async Generation

When the decorated function is asynchronous, the generated middleware implements:

```python
awrap_model_call
```

When the decorated function is synchronous, the generated middleware implements both:

```python
wrap_model_call
awrap_model_call
```

The asynchronous wrapper calls the synchronous prompt function directly and then awaits the model handler.

## Example

```python
@dynamic_prompt
def context_prompt(
    request: ModelRequest
) -> str:
    user_name = (
        request.runtime.context
        .get("user_name", "User")
    )

    return (
        "You are helping "
        f"{user_name}."
    )
```

# `wrap_model_call`

Decorator that converts a standalone model-wrapper function into middleware.

```python
wrap_model_call(
    func=None,
    *,
    state_schema: type[StateT] | None = None,
    tools: list[BaseTool] | None = None,
    name: str | None = None
)
```

## Decorated Function Signature

### Synchronous

```python
def wrapper(
    request: ModelRequest[ContextT],
    handler: Callable[
        [ModelRequest[ContextT]],
        ModelResponse[ResponseT]
    ]
) -> ModelCallResult:
    ...
```

### Asynchronous

```python
async def wrapper(
    request: ModelRequest[ContextT],
    handler: Callable[
        [ModelRequest[ContextT]],
        Awaitable[
            ModelResponse[ResponseT]
        ]
    ]
) -> ModelCallResult:
    ...
```

## Generated Middleware

A synchronous function creates a middleware class implementing only:

```python
wrap_model_call
```

An asynchronous function creates one implementing only:

```python
awrap_model_call
```

## Example: Retry

```python
@wrap_model_call
def retry_model(
    request,
    handler
):
    for attempt in range(3):
        try:
            return handler(request)
        except Exception:
            if attempt == 2:
                raise
```

## Example: Fallback

```python
@wrap_model_call
def fallback(
    request,
    handler
):
    try:
        return handler(request)
    except Exception:
        return handler(
            request.override(
                model=fallback_model
            )
        )
```

# `wrap_tool_call`

Decorator that converts a standalone tool-wrapper function into middleware.

```python
wrap_tool_call(
    func=None,
    *,
    tools: list[BaseTool] | None = None,
    name: str | None = None
)
```

## Decorated Function Signature

### Synchronous

```python
def wrapper(
    request: ToolCallRequest,
    handler: Callable[
        [ToolCallRequest],
        ToolMessage | Command[Any]
    ]
) -> ToolMessage | Command[Any]:
    ...
```

### Asynchronous

```python
async def wrapper(
    request: ToolCallRequest,
    handler: Callable[
        [ToolCallRequest],
        Awaitable[
            ToolMessage | Command[Any]
        ]
    ]
) -> ToolMessage | Command[Any]:
    ...
```

## Generated Middleware

* A sync function implements only `wrap_tool_call`.
* An async function implements only `awrap_tool_call`.
* The generated middleware always uses `AgentState` as its state schema.
* Additional tools may be supplied through the `tools` parameter.

## Example

```python
@wrap_tool_call
def double_value(
    request,
    handler
):
    modified_call = {
        **request.tool_call,
        "args": {
            **request.tool_call["args"],
            "value": (
                request.tool_call[
                    "args"
                ]["value"] * 2
            )
        }
    }

    return handler(
        request.override(
            tool_call=modified_call
        )
    )
```

# `ToolCallRequest`

Re-exported from:

```python
langgraph.prebuilt.tool_node
```

It represents one tool-execution request and provides access to:

* `tool_call`
* Bound `tool`
* Agent `state`
* `runtime`
* Request override functionality

The exact class implementation belongs to LangGraph rather than this module.

# `ToolCallWrapper`

Re-exported from:

```python
langgraph.prebuilt.tool_node
```

It represents the callable wrapper type used around tool execution.

The exact implementation belongs to LangGraph.

# Middleware Composition

Model and tool wrappers compose as nested handlers.

For:

```python
middleware=[
    outer,
    inner
]
```

the effective model-call flow is:

```text
outer.wrap_model_call
        |
        v
inner.wrap_model_call
        |
        v
actual model call
        |
        v
inner receives result
        |
        v
outer receives result
```

The same outermost-first composition applies to tool wrappers.

# Important Sync/Async Rule

A middleware must implement the execution mode used by the agent.

For example, defining only:

```python
awrap_model_call
```

and then calling:

```python
agent.invoke(...)
```

causes the synchronous base method to raise `NotImplementedError`.

Likewise, defining only:

```python
wrap_tool_call
```

and using:

```python
await agent.ainvoke(...)
```

causes the asynchronous base method to raise `NotImplementedError`.

Use the matching method or invocation style.

# Exports

```python
__all__ = [
    "AgentMiddleware",
    "AgentState",
    "ContextT",
    "ExtendedModelResponse",
    "InputAgentState",
    "ModelCallResult",
    "ModelRequest",
    "ModelResponse",
    "OmitFromSchema",
    "OutputAgentState",
    "ResponseT",
    "StateT_co",
    "ToolCallRequest",
    "ToolCallWrapper",
    "after_agent",
    "after_model",
    "before_agent",
    "before_model",
    "dynamic_prompt",
    "hook_config",
    "wrap_tool_call",
]
```

At this pinned revision, `wrap_model_call` is defined in the module but is not listed in `__all__`.

The following useful values are also defined but are not included in `__all__`:

```text
JumpTo
OmitFromInput
OmitFromOutput
PrivateStateAttr
StateT
StateT_contra
```

# Source

This reference follows the pinned LangChain source:

```text
libs/langchain_v1/langchain/agents/middleware/types.py
Commit: 42f8f79293cfb7589e5bc1d74a8ae4dfd0bf15e3
```